In [101]:
# Install required libraries

!pip install -q -U google-genai pypdf

In [102]:
# Import required libraries and connect to Gemini

import numpy as np
import time
from pypdf import PdfReader
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("Ragproject"))

print("Gemini client connected successfully")

Gemini client connected successfully


In [103]:
# Upload the TMB KYC/AML policy PDF

from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded file:", pdf_file)

Saving TMB_bank_policy.pdf to TMB_bank_policy (3).pdf
Uploaded file: TMB_bank_policy (3).pdf


In [104]:
# Define the text chunking function

def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

print("Chunking function defined")

Chunking function defined


In [105]:
import io
from pypdf import PdfReader

# Read the PDF content into the 'text' variable
text = ""
reader = PdfReader(io.BytesIO(uploaded[pdf_file]))
for page in reader.pages:
    text += page.extract_text()

print("Total characters:", len(text))

# Split the document into 800-character chunks
chunks = chunk_text(text, chunk_size=800, overlap=150)

print("Number of chunks:", len(chunks))

# Truncate output preview to 150 characters for clean GitHub rendering
preview = chunks[0][:150].replace("\n", " ")
print(f"\nFirst full chunk (Preview):\n{preview}...\n")

Total characters: 7030
Number of chunks: 11

First full chunk (Preview):
KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridged Version                     ...



In [106]:
# Define the embedding model and embedding function

EMP_MODEL = "gemini-embedding-001"

def embed_chunks(t):
    result = client.models.embed_content(
        model=EMP_MODEL,
        contents=t,
        config=types.EmbedContentConfig(
            output_dimensionality=768
        )
    )

    return np.array(result.embeddings[0].values)

print("Embedding function defined")

Embedding function defined


In [107]:
chunk_embeddings = []

for i, ch in enumerate(chunks):
    chunk_embeddings.append(embed_chunks(ch))

    print(f"Embedded chunks {i+1}/{len(chunks)}")

    time.sleep(2)

chunk_embeddings = np.array(chunk_embeddings)
print(f"Total chunks embedded: {len(chunk_embeddings)}")

Embedded chunks 1/11
Embedded chunks 2/11
Embedded chunks 3/11
Embedded chunks 4/11
Embedded chunks 5/11
Embedded chunks 6/11
Embedded chunks 7/11
Embedded chunks 8/11
Embedded chunks 9/11
Embedded chunks 10/11
Embedded chunks 11/11
Total chunks embedded: 11


In [108]:
# Normalize the chunk embeddings for similarity search

chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings,
    axis=1,
    keepdims=True
)

print(
    "Stored",
    len(chunks),
    "chunks in vector store",
    chunk_embeddings.shape
)

Stored 11 chunks in vector store (11, 768)


In [109]:
# Retrieve the top 3 most relevant chunks

def retrieve(question, k=3):

    query_embedding = embed_chunks(question)

    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    scores = chunk_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "chunk": chunks[index],
            "score": float(scores[index]),
            "index": int(index)
        })

    return results

print("Retrieval function defined")

Retrieval function defined


In [110]:
# Test retrieval before connecting the generation model

question = "What is the primary objective of the KYC/AML Policy"
results = retrieve(question, k=3)

print(f"--- Search Results for: '{question}' ---\n")

for rank, item in enumerate(results, 1):
    # Truncate text to 150 characters and remove line breaks for compact viewing
    preview = item["chunk"][:150].replace("\n", " ")

    print(f"Rank {rank} | Chunk Index: {item['index']} | Score: {item['score']:.4f}")
    print(f"Preview: {preview}...\n")


--- Search Results for: 'What is the primary objective of the KYC/AML Policy' ---

Rank 1 | Chunk Index: 3 | Score: 0.7289
Preview:  of the KYC/AML Policy of the Bank is to prevent the Bank from being used,  intentionally or unintentionally, by criminal elements for money launderin...

Rank 2 | Chunk Index: 2 | Score: 0.7275
Preview: o the integrity of the international financial  system. India, being a member of FATF, is committed to upholding measures to protect the integrity of ...

Rank 3 | Chunk Index: 0 | Score: 0.7034
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridged Version                     ...



In [111]:
# Define the Gemini model for answering questions

MODEL = "gemini-3.1-flash-lite"

print("Generation model:", MODEL)

Generation model: gemini-3.1-flash-lite


In [112]:
# Generate an answer using only the retrieved chunks

def ask(question, k=3):

    results = retrieve(question, k)

    context = "\n\n".join(
        [
            f"Chunk {r['index']}:\n{r['chunk']}"
            for r in results
        ]
    )

    prompt = f"""
Answer the question using ONLY the information provided in the context.

If the answer cannot be found in the context, clearly say:
"I cannot find the answer in the provided document."

Do not use outside knowledge.
Do not guess.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text, results

print("RAG answer function defined")

RAG answer function defined


In [113]:
# Display the answer along with its source chunks (truncated) and scores
def ask_and_show(question):
    answer, results = ask(question, k=3)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCE CHUNKS:")

    for i, result in enumerate(results, 1):
        # Truncate to first 120 characters to keep GitHub output clean
        preview = result["chunk"][:120].replace("\n", " ")

        print("\n" + "-" * 80)
        print(f"Source {i}")
        print("Chunk index:", result["index"])
        print("Similarity score:", round(result["score"], 4))
        print("Preview:", preview + "...\n")

In [114]:
# Test an in-document question about the policy objective

ask_and_show(
    "What is the primary objective of the KYC/AML Policy?"
)

QUESTION:
What is the primary objective of the KYC/AML Policy?

ANSWER:
The primary objective of the KYC/AML Policy of the Bank is to prevent the Bank from being used, intentionally or unintentionally, by criminal elements for money laundering or financing of terrorism.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 3
Similarity score: 0.7284
Preview:  of the KYC/AML Policy of the Bank is to prevent the Bank from being used,  intentionally or unintentionally, by crimina...


--------------------------------------------------------------------------------
Source 2
Chunk index: 2
Similarity score: 0.7244
Preview: o the integrity of the international financial  system. India, being a member of FATF, is committed to upholding measure...


--------------------------------------------------------------------------------
Source 3
Chunk index: 0
Similarity score: 0.7026
Preview: KYC/AML Policy – Abridged Version Page 1 of

In [115]:
ask_and_show(
    "How often is the KYC policy document reviewed by the Board of Directors?"
)

QUESTION:
How often is the KYC policy document reviewed by the Board of Directors?

ANSWER:
The KYC policy document is reviewed by the Board of Directors annually.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 3
Similarity score: 0.7396
Preview:  of the KYC/AML Policy of the Bank is to prevent the Bank from being used,  intentionally or unintentionally, by crimina...


--------------------------------------------------------------------------------
Source 2
Chunk index: 7
Similarity score: 0.6872
Preview: ls (entities), identification information of individuals, who are  Proprietor(s) / Partner(s) / Beneficial Owner(s) /Aut...


--------------------------------------------------------------------------------
Source 3
Chunk index: 10
Similarity score: 0.6678
Preview: igence  c) Simplified due diligence  d) Identification of Beneficial Owner  e) CDD Procedure and sharing KYC information...



In [116]:
ask_and_show(
    "What year was the Prevention of Money-Laundering Act enacted?"
)

QUESTION:
What year was the Prevention of Money-Laundering Act enacted?

ANSWER:
2002

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 1
Similarity score: 0.6344
Preview: n procedures while undertaking a transaction either by establishing an account -based  relationship or otherwise and mon...


--------------------------------------------------------------------------------
Source 2
Chunk index: 2
Similarity score: 0.5819
Preview: o the integrity of the international financial  system. India, being a member of FATF, is committed to upholding measure...


--------------------------------------------------------------------------------
Source 3
Chunk index: 0
Similarity score: 0.5805
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridge...



In [117]:
# Test an out-of-document question

ask_and_show(
    "Who is the current Governor of the Reserve Bank of India?"
)

QUESTION:
Who is the current Governor of the Reserve Bank of India?

ANSWER:
I cannot find the answer in the provided document.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 2
Similarity score: 0.5137
Preview: o the integrity of the international financial  system. India, being a member of FATF, is committed to upholding measure...


--------------------------------------------------------------------------------
Source 2
Chunk index: 6
Similarity score: 0.5091
Preview:  the equivalent e-document thereof as may be required by the Business / Ope ration Units,  to complete requisite CDD and...


--------------------------------------------------------------------------------
Source 3
Chunk index: 1
Similarity score: 0.5083
Preview: n procedures while undertaking a transaction either by establishing an account -based  relationship or otherwise and mon...



In [118]:
ask_and_show(
    "What is India's current GDP growth rate?"
)

QUESTION:
What is India's current GDP growth rate?

ANSWER:
I cannot find the answer in the provided document.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 1
Similarity score: 0.4888
Preview: n procedures while undertaking a transaction either by establishing an account -based  relationship or otherwise and mon...


--------------------------------------------------------------------------------
Source 2
Chunk index: 6
Similarity score: 0.4873
Preview:  the equivalent e-document thereof as may be required by the Business / Ope ration Units,  to complete requisite CDD and...


--------------------------------------------------------------------------------
Source 3
Chunk index: 10
Similarity score: 0.4789
Preview: igence  c) Simplified due diligence  d) Identification of Beneficial Owner  e) CDD Procedure and sharing KYC information...



In [119]:
ask_and_show(
    "What is the population of India?"
)

QUESTION:
What is the population of India?

ANSWER:
I cannot find the answer in the provided document.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 0
Similarity score: 0.4839
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridge...


--------------------------------------------------------------------------------
Source 2
Chunk index: 6
Similarity score: 0.4723
Preview:  the equivalent e-document thereof as may be required by the Business / Ope ration Units,  to complete requisite CDD and...


--------------------------------------------------------------------------------
Source 3
Chunk index: 1
Similarity score: 0.4651
Preview: n procedures while undertaking a transaction either by establishing an account -based  relationship or otherwise and mon...



### Observation

The RAG system correctly retrieved relevant chunks for the in-document questions and generated answers based on the TMB KYC/AML Policy. The out-of-document questions were handled using the refusal instruction when the required information was not present in the retrieved context.

In [120]:
# Create larger 3000-character chunks for comparison
large_chunks = chunk_text(text, chunk_size=3000, overlap=150)

print("Number of large chunks:", len(large_chunks))

# Truncate output preview to 150 characters for clean GitHub rendering
preview = large_chunks[0][:150].replace("\n", " ")
print(f"\nFirst large chunk (Preview):\n{preview}...\n")

Number of large chunks: 3

First large chunk (Preview):
KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridged Version                     ...



In [121]:
# Generate embeddings for the 3000-character chunks

large_embeddings = []

for i, ch in enumerate(large_chunks):

    large_embeddings.append(embed_chunks(ch))

    print(f"Embedded large chunks {i+1}/{len(large_chunks)}")

    time.sleep(2)

large_embeddings = np.array(large_embeddings)

print("\nLarge embedding shape:", large_embeddings.shape)

Embedded large chunks 1/3
Embedded large chunks 2/3
Embedded large chunks 3/3

Large embedding shape: (3, 768)


In [122]:
# Normalize the large chunk embeddings

large_embeddings = large_embeddings / np.linalg.norm(
    large_embeddings,
    axis=1,
    keepdims=True
)

print(
    "Stored",
    len(large_chunks),
    "large chunks",
    large_embeddings.shape
)

Stored 3 large chunks (3, 768)


In [123]:
# Retrieve the top 3 chunks from the 3000-character chunk set

def retrieve_large(question, k=3):

    query_embedding = embed_chunks(question)

    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    scores = large_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in top_indices:

        results.append({
            "chunk": large_chunks[index],
            "score": float(scores[index]),
            "index": int(index)
        })

    return results

print("Large-chunk retrieval function defined")

Large-chunk retrieval function defined


In [124]:
# Generate answers using the 3000-character chunks

def ask_large(question, k=3):

    results = retrieve_large(question, k)

    context = "\n\n".join(
        [
            f"Chunk {r['index']}:\n{r['chunk']}"
            for r in results
        ]
    )

    prompt = f"""
Answer the question using ONLY the information provided in the context.

If the answer cannot be found in the context, clearly say:
"I cannot find the answer in the provided document."

Do not use outside knowledge.
Do not guess.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text, results

print("Large-chunk answer function defined")

Large-chunk answer function defined


In [125]:
# Display large-chunk answers with truncated source chunks and scores
def ask_and_show_large(question):
    answer, results = ask_large(question, k=3)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCE CHUNKS:")

    for i, result in enumerate(results, 1):
        # Truncate text to 120 characters to keep GitHub display clean
        preview = result["chunk"][:120].replace("\n", " ")

        print("\n" + "-" * 80)
        print(f"Source {i}")
        print("Chunk index:", result["index"])
        print("Similarity score:", round(result["score"], 4))
        print("Preview:", preview + "...\n")

In [126]:
# Test Question 1 with 3000-character chunks

ask_and_show_large(
    "What is the primary objective of the KYC/AML Policy?"
)

QUESTION:
What is the primary objective of the KYC/AML Policy?

ANSWER:
The primary objective of the KYC/AML Policy of the Bank is to prevent the Bank from being used, intentionally or unintentionally, by criminal elements for money laundering or financing of terrorism.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 0
Similarity score: 0.7038
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridge...


--------------------------------------------------------------------------------
Source 2
Chunk index: 2
Similarity score: 0.6644
Preview: purpose without the express permission of the customer.  (c) While considering the requests for data/information from Go...


--------------------------------------------------------------------------------
Source 3
Chunk index: 1
Similarity score: 0.6542
Preview: al while establishing an account -based  re

In [127]:
ask_and_show_large(
    "How often is the KYC policy document reviewed by the Board of Directors?"
)

QUESTION:
How often is the KYC policy document reviewed by the Board of Directors?

ANSWER:
The KYC policy document is reviewed by the Board of Directors annually.

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 0
Similarity score: 0.6854
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridge...


--------------------------------------------------------------------------------
Source 2
Chunk index: 1
Similarity score: 0.6726
Preview: al while establishing an account -based  relationship or while dealing with the individual who is a beneficial owner, au...


--------------------------------------------------------------------------------
Source 3
Chunk index: 2
Similarity score: 0.655
Preview: purpose without the express permission of the customer.  (c) While considering the requests for data/information from Go...



In [128]:
ask_and_show_large(
    "What year was the Prevention of Money-Laundering Act enacted?"
)

QUESTION:
What year was the Prevention of Money-Laundering Act enacted?

ANSWER:
2002

SOURCE CHUNKS:

--------------------------------------------------------------------------------
Source 1
Chunk index: 0
Similarity score: 0.5742
Preview: KYC/AML Policy – Abridged Version Page 1 of 4                                  KNOW YOUR CUSTOMER (KYC) NORMS    Abridge...


--------------------------------------------------------------------------------
Source 2
Chunk index: 1
Similarity score: 0.5529
Preview: al while establishing an account -based  relationship or while dealing with the individual who is a beneficial owner, au...


--------------------------------------------------------------------------------
Source 3
Chunk index: 2
Similarity score: 0.5498
Preview: purpose without the express permission of the customer.  (c) While considering the requests for data/information from Go...



### 800 vs 3000 Character Chunk Comparison

| Metric / Test Question | 800-Character Chunks (11 Total) | 3000-Character Chunks (3 Total) | Performance Winner |
|---|---|---|---|
| **Top Chunk Score: Primary Objective** | **0.7284** (Chunk #3) — *Correct* | **0.7038** (Chunk #0) — *Correct* | **800-Character Chunks** (+0.0246 higher similarity) |
| **Top Chunk Score: Board Review Frequency** | **0.7396** (Chunk #3) — *Correct* | **0.6854** (Chunk #0) — *Correct* | **800-Character Chunks** (+0.0542 higher similarity) |
| **Top Chunk Score: Enactment Year of PMLA** | **0.6344** (Chunk #1) — *Correct* | **0.5742** (Chunk #0) — *Correct* | **800-Character Chunks** (+0.0602 higher similarity) |
| **Out-of-Document: RBI Governor** | Refused — *"I cannot find the answer..."* | Refused — *"I cannot find the answer..."* | **Tie** (Both strict guardrails succeeded) |
| **Out-of-Document: India GDP Growth** | Refused — *"I cannot find the answer..."* | Refused — *"I cannot find the answer..."* | **Tie** (Both strict guardrails succeeded) |
| **Retrieval Precision & Noise** | **High** (focused context window) | **Low** (includes unrelated sections) | **800-Character Chunks** |

### Observation

With 800-character chunks, the document was divided into 11 smaller chunks, while 3000-character chunks produced only 3 larger chunks. The 800-character chunks generally produced higher similarity scores for the three tested questions. Both chunk sizes produced correct answers because the required information was present in the retrieved context. The smaller chunks provided more focused retrieval, while the larger chunks provided more surrounding context. Overall, the 800-character chunk size performed slightly better for this document based on the observed scores.